# Bronze — S&P 500 tracker prices

`landing.index_prices_raw` → `bronze.index_prices`. Every column cast to STRING.

This table carries **12 columns against the trust table's 11**: Yahoo returns
`Capital_Gains` for these instruments and not for the trusts. Same notebook logic, one
more column, no code change — which is the argument for casting the live schema instead
of naming columns in advance.

Expected: **915 rows across 3 symbols**, matching Landing exactly.

In [0]:
CATALOG = "`index-vs-trust-pipeline`"
SOURCE = f"{CATALOG}.landing.index_prices_raw"
TARGET = f"{CATALOG}.bronze.index_prices"

In [0]:
src_columns = spark.table(SOURCE).columns

# Cast whatever arrived. Yahoo's column set moves between instruments -- Capital_Gains
# comes back here and not for the trusts -- so a hardcoded list would drop it.
cast_list = ",\n  ".join(f"CAST(`{c}` AS STRING) AS `{c}`" for c in src_columns)
sql = f"CREATE OR REPLACE TABLE {TARGET} AS\nSELECT\n  {cast_list}\nFROM {SOURCE}"

print(f"{len(src_columns)} columns found: {src_columns}\n")
print(sql)

spark.sql(sql)
print(f"\nwrote {TARGET}")

## Verification

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.landing.index_prices_raw) AS landing_rows,
  (SELECT COUNT(*) FROM `index-vs-trust-pipeline`.bronze.index_prices)      AS bronze_rows,
  (SELECT COUNT(DISTINCT symbol) FROM `index-vs-trust-pipeline`.bronze.index_prices) AS symbols;

Expect **915 / 915 / 3 symbols** — SPY, IVV, VOO. SPLG never returned data, so it has no
rows here either.

In [0]:
%sql
-- The contract, plus proof Capital_Gains came through rather than being dropped.
SELECT COUNT(*)                                               AS columns_total,
       SUM(CASE WHEN data_type <> 'STRING' THEN 1 ELSE 0 END)  AS not_string,
       SUM(CASE WHEN column_name = 'Capital_Gains' THEN 1 ELSE 0 END) AS has_capital_gains
FROM `index-vs-trust-pipeline`.information_schema.columns
WHERE table_schema = 'bronze' AND table_name = 'index_prices';

Expect **12 columns, 0 not_string, has_capital_gains = 1**.

That last one is the check that matters. A hardcoded column list copied from the trust
notebook would have returned 11 and 0 here, and `Capital_Gains` would have vanished
without any error being raised.